In [2]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
# Set the directory path containing the files
directory_path <- "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/"
# Initialize an empty data frame to store the results
result <- data.frame()


In [4]:
chrom <- 22

test_path <- paste0(directory_path, "chr", chrom, ".rfmix.Q")
print(test_path)

[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr22.rfmix.Q"


In [5]:
test_data <- read.table(test_path, header = F, sep = "\t")

In [6]:
colnames(test_data) <- c("IID", "CEU", "YRI")

In [7]:
head(test_data)

,IID,CEU,YRI
,<chr>,<dbl>,<dbl>
1,SF0000003_SP0000003,0.91714,0.08286
2,SF0000017_SP0000017,1.00000,0.00000
3,SF0000027_SP0000027,0.97943,0.02057
4,SF0000027_SP0000034,1.00000,0.00000
5,SF0000027_SP0000051,1.00000,0.00000
6,SF0000027_SP0000063,1.00000,0.00000


In [8]:
range(test_data$CEU)

[1] 0.59371 1.00000

In [9]:
# Loop through files from ace_chr1 to ace_chr22
for (chr in 1:22) {
    

  # Construct the file path
  file_path <- paste0(directory_path, "chr", chr, ".rfmix.Q")
  print(file_path)
  # Read the data from the current file
  data <- read.table(file_path, header = F, sep = "\t")
  colnames(data) <- c("IID", "CEU", "YRI")
  # Combine the results for each ancestry (ANC1 and ANC2)
  result <- rbind(result, data)
}


[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr1.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr2.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr3.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr4.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr5.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr6.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr7.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr8.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr9.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr10.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr11.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr12.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/chr13.rfmix.Q"
[1] "/u/project/geschwind/shared/PRS/20240221_SPARK_RFMix/ch

In [10]:
head(result)

,IID,CEU,YRI
,<chr>,<dbl>,<dbl>
1,SF0000003_SP0000003,0.76869,0.23131
2,SF0000017_SP0000017,0.89637,0.10363
3,SF0000027_SP0000027,0.88714,0.11286
4,SF0000027_SP0000034,0.89566,0.10434
5,SF0000027_SP0000051,0.90304,0.09696
6,SF0000027_SP0000063,0.88520,0.11480


In [11]:
result[result$IID == "SF0000003_SP0000003",]

,IID,CEU,YRI
,<chr>,<dbl>,<dbl>
1,SF0000003_SP0000003,0.76869,0.23131
69488,SF0000003_SP0000003,0.76497,0.23503
138975,SF0000003_SP0000003,0.91646,0.08354
208462,SF0000003_SP0000003,0.86327,0.13673
277949,SF0000003_SP0000003,0.90448,0.09552
347436,SF0000003_SP0000003,0.96072,0.03928
416923,SF0000003_SP0000003,0.96963,0.03037
486410,SF0000003_SP0000003,0.83262,0.16738
555897,SF0000003_SP0000003,0.71947,0.28053


In [12]:
# Sum the scores across all chromosomes for each individual by ancestry
final_result <- result %>%
  group_by(IID) %>%
  summarize(
    CEU_avg = mean(CEU),
    YRI_avg = mean(YRI)
  )

In [13]:
head(final_result)


IID,CEU_avg,YRI_avg
<chr>,<dbl>,<dbl>
SF0000003_SP0000003,0.8416727,0.1583273
SF0000017_SP0000017,0.8866282,0.1133718
SF0000027_SP0000027,0.8944382,0.1055618
SF0000027_SP0000034,0.8989545,0.1010455
SF0000027_SP0000051,0.8953018,0.1046982
SF0000027_SP0000063,0.8997091,0.1002909


In [6]:
final_result$sum <- final_result$CEU_avg + final_result$YRI_avg

In [7]:
range(final_result$sum)

[1] 1 1

In [8]:
write_csv(final_result[,c("IID", "CEU_avg", "YRI_avg")], "spark_anc_part_prop.csv")

In [14]:
range(final_result$CEU_avg)

[1] 0.4559945 0.9268186